# How to Use the Tiles API for the xarray backend

The `/xarray/tiles` API is used to generate map tiles via [the tiles OGC standard](https://www.ogc.org/standards/ogcapi-tiles/).

This notebook demonstrates how to use the tiles API for the [GHRSST Level 4 MUR Global Foundation Sea Surface Temperature Analysis (v4.1)](https://cmr.earthdata.nasa.gov/search/concepts/C1996881146-POCLOUD) product.

## Setup

In [ ]:
from datetime import datetime, timezone

import earthaccess
import httpx
from folium import Map, TileLayer

titiler_endpoint = "https://staging.openveda.cloud/api/titiler-cmr"  # staging endpoint

## Identify the dataset

You can find the MUR SST dataset using the `earthaccess.search_datasets` function.

In [ ]:
datasets = earthaccess.search_datasets(doi="10.5067/GHGMR-4FJ04")
ds = datasets[0]

collection_concept_id = ds["meta"]["concept-id"]
print("Collection Concept-Id: ", collection_concept_id)

print("Abstract: ", ds["umm"]["Abstract"])

## Explore the collection using the `/compatibility` endpoint

See [How to use the Compatibility API Endpoint](../compatibility_api_example) for how the compatiliby endpoint can be used to identify variable, datetime and rescale parameters.

## Define a query for titiler-cmr

To use titiler-cmr's endpoints for a NetCDF dataset like this we need to define a date range for the CMR query and a `variable` to analyze.

In [ ]:
variable = "sea_ice_fraction"
datetime_ = datetime(2024, 10, 10, tzinfo=timezone.utc).isoformat()

## Display tiles in an interactive map

The `/tilejson.json` endpoint will provide a parameterized `xyz` tile URL that can be added to an interactive map.

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params=(
        ("collection_concept_id", collection_concept_id),
        # Temporal range in form of `start_date/end_date`
        ("temporal", datetime_),
        ("variables", variable),
        # We need to set min/max zoom because we don't want to use lowerzoom level (e.g 0)
        # which will results in useless large scale query
        ("minzoom", 2),
        ("maxzoom", 13),
        ("rescale", "0,1"),
        ("colormap_name", "blues_r"),
    ),
    timeout=None,
).json()

print(r)

In [ ]:
bounds = r["bounds"]
m = Map(location=(70, -40), zoom_start=3)

TileLayer(
    tiles=r["tiles"][0],
    opacity=1,
    attr="NASA",
).add_to(m)
m

## Use the `expression` parameter to combine multiple variables

The `xarray` tiling endpoints can accept a list of variables to use when creating the output images. When combined with the `expression` parameter this is very powerful!

The following example shows how to create an RGB composite image using the HHHH and HVHV variables from a NISAR GCOV granule.

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params=(
        ("collection_concept_id", "C3622214170-ASF"),
        (
            "granule_ur",
            "NISAR_L2_PR_GCOV_009_026_A_015_4005_DHDH_A_20251229T234707_20251229T234741_X05009_N_F_J_001",
        ),
        ("group", "/science/LSAR/GCOV/grids/frequencyA"),
        ("variables", "HHHH"),  # b1
        ("variables", "HVHV"),  # b2
        ("expression", "10 * log10(b1); 10 * log10(b2); 10 * log10(b1)"),
        ("rescale", "-20, 0"),
        ("rescale", "-30, -5"),
        ("rescale", "-15, 0"),
    ),
    timeout=None,
).json()

m = Map(location=(r["center"][1], r["center"][0]), zoom_start=9)

TileLayer(
    tiles=r["tiles"][0],
    opacity=1,
    attr="NASA/ISRO",
).add_to(m)
m